# Comparative Study: All Feature Extraction Methods

This notebook compares different feature extraction methods for image classification:
1. **CNN Features** (FC layer removed) + SVM
2. **ViT Features** ([CLS] token) + SVM
3. **BoVW with CNN Descriptors** + SVM
4. **BoVW with SIFT Descriptors** + SVM

## Methodology
- All methods use SVM with RBF kernel for fair comparison
- Same train/test split (80/20) with random_state=42
- Same dataset: UCMerced Land Use Dataset (21 classes)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import time

from data import DataLoader
from config import ExperimentConfig
from features import FeatureExtractor
from vocabulary import VocabularyBuilder
from classifier import Classifier

# Set style for better plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)


## 1. Load Data


In [ ]:
# Load data once for all experiments
config_base = ExperimentConfig(data_dir="UCMerced_LandUse")
data_loader = DataLoader(config_base)
images, labels, class_names = data_loader.load_images_and_labels()

print(f"Dataset Statistics:")
print(f"  Total images: {len(images)}")
print(f"  Number of classes: {len(class_names)}")
print(f"  Classes: {class_names}")

# Store results for comparison
results = []


## 2. Method 1: CNN Features + SVM


In [ ]:
print("="*60)
print("METHOD 1: CNN Features (FC removed) + SVM")
print("="*60)

config = ExperimentConfig(
    data_dir="UCMerced_LandUse",
    classifier_type="svm",
    svm_kernel="rbf",
    svm_C=1.0,
    use_feature_scaling=True,
    cnn_model="resnet18"
)

# Extract features
feature_extractor = FeatureExtractor(config)
start_time = time.time()
features = feature_extractor.deep_nn_feature_extraction(images, model_name=config.cnn_model)
extraction_time = time.time() - start_time

# Split
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train
clf = svm.SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
train_start = time.time()
clf.fit(X_train, y_train)
train_time = time.time() - train_start

# Evaluate
y_pred = clf.predict(X_test)
accuracy = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = metrics.recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = metrics.f1_score(y_test, y_pred, average='weighted', zero_division=0)

results.append({
    'Method': 'CNN Features (ResNet18)',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Extraction Time': extraction_time,
    'Training Time': train_time,
    'Feature Dim': features.shape[1]
})

print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
print(f"Extraction: {extraction_time:.2f}s, Training: {train_time:.2f}s")


## 3. Method 2: ViT Features + SVM


In [ ]:
print("\n" + "="*60)
print("METHOD 2: ViT Features ([CLS] token) + SVM")
print("="*60)

config = ExperimentConfig(
    data_dir="UCMerced_LandUse",
    classifier_type="svm",
    svm_kernel="rbf",
    svm_C=1.0,
    use_feature_scaling=True,
    vit_model_name="google/vit-base-patch16-224"
)

# Extract features
feature_extractor = FeatureExtractor(config)
start_time = time.time()
features = feature_extractor.vit_feature_extraction(images, model_name=config.vit_model_name)
extraction_time = time.time() - start_time

# Split
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train
clf = svm.SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
train_start = time.time()
clf.fit(X_train, y_train)
train_time = time.time() - train_start

# Evaluate
y_pred = clf.predict(X_test)
accuracy = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = metrics.recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = metrics.f1_score(y_test, y_pred, average='weighted', zero_division=0)

results.append({
    'Method': 'ViT Features',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Extraction Time': extraction_time,
    'Training Time': train_time,
    'Feature Dim': features.shape[1]
})

print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
print(f"Extraction: {extraction_time:.2f}s, Training: {train_time:.2f}s")


## 4. Method 3: BoVW with CNN Descriptors + SVM


In [ ]:
print("\n" + "="*60)
print("METHOD 3: BoVW (CNN Descriptors) + SVM")
print("="*60)

config = ExperimentConfig(
    data_dir="UCMerced_LandUse",
    classifier_type="svm",
    svm_kernel="rbf",
    svm_C=1.0,
    dsc_method="cnn",
    cnn_model="vgg16",
    vocab_size=150,
    use_minibatch_kmeans=True,
    normalize_histograms=True
)

# Extract descriptors
feature_extractor = FeatureExtractor(config)
start_time = time.time()
image_descriptors, all_descriptors = feature_extractor.extract_descriptors(images)
extraction_time = time.time() - start_time

# Build vocabulary
vocab_builder = VocabularyBuilder(config)
vocab_start = time.time()
vocabulary = vocab_builder.build_vocabulary(all_descriptors)
vocab_time = time.time() - vocab_start

# Compute histograms
hist_start = time.time()
histograms = vocab_builder.compute_histograms(image_descriptors)
hist_time = time.time() - hist_start

total_extraction_time = extraction_time + vocab_time + hist_time

# Split
X_train, X_test, y_train, y_test = train_test_split(
    histograms, labels, test_size=0.2, random_state=42, stratify=labels
)

# Train
clf = Classifier(config)
train_start = time.time()
clf.train(X_train, y_train)
train_time = time.time() - train_start

# Evaluate
metrics_dict = clf.evaluate(X_test, y_test, class_names)
accuracy = metrics_dict['accuracy']
precision = metrics_dict['precision']
recall = metrics_dict['recall']
f1 = metrics_dict['f1_score']

results.append({
    'Method': 'BoVW (CNN Descriptors)',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Extraction Time': total_extraction_time,
    'Training Time': train_time,
    'Feature Dim': histograms.shape[1]
})

print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
print(f"Extraction: {total_extraction_time:.2f}s, Training: {train_time:.2f}s")


## 5. Method 4: BoVW with SIFT Descriptors + SVM


In [ ]:
print("\n" + "="*60)
print("METHOD 4: BoVW (SIFT Descriptors) + SVM")
print("="*60)

config = ExperimentConfig(
    data_dir="UCMerced_LandUse",
    classifier_type="svm",
    svm_kernel="rbf",
    svm_C=1.0,
    dsc_method="sift",
    sift_n_features=0,
    vocab_size=150,
    use_minibatch_kmeans=True,
    normalize_histograms=True
)

# Extract descriptors
feature_extractor = FeatureExtractor(config)
start_time = time.time()
image_descriptors, all_descriptors = feature_extractor.extract_descriptors(images)
extraction_time = time.time() - start_time

# Build vocabulary
vocab_builder = VocabularyBuilder(config)
vocab_start = time.time()
vocabulary = vocab_builder.build_vocabulary(all_descriptors)
vocab_time = time.time() - vocab_start

# Compute histograms
hist_start = time.time()
histograms = vocab_builder.compute_histograms(image_descriptors)
hist_time = time.time() - hist_start

total_extraction_time = extraction_time + vocab_time + hist_time

# Split
X_train, X_test, y_train, y_test = train_test_split(
    histograms, labels, test_size=0.2, random_state=42, stratify=labels
)

# Train
clf = Classifier(config)
train_start = time.time()
clf.train(X_train, y_train)
train_time = time.time() - train_start

# Evaluate
metrics_dict = clf.evaluate(X_test, y_test, class_names)
accuracy = metrics_dict['accuracy']
precision = metrics_dict['precision']
recall = metrics_dict['recall']
f1 = metrics_dict['f1_score']

results.append({
    'Method': 'BoVW (SIFT Descriptors)',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Extraction Time': total_extraction_time,
    'Training Time': train_time,
    'Feature Dim': histograms.shape[1]
})

print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
print(f"Extraction: {total_extraction_time:.2f}s, Training: {train_time:.2f}s")


## 6. Comparison Results

### 6.1 Results Table


In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.round(4)

print("="*80)
print("COMPARATIVE RESULTS")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)


### 6.2 Performance Metrics Comparison


In [ ]:
# Plot comparison of all metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    bars = ax.barh(results_df['Method'], results_df[metric], color=colors[idx], alpha=0.8)
    ax.set_xlabel(metric, fontsize=12)
    ax.set_title(f'{metric} Comparison', fontsize=13, fontweight='bold')
    ax.set_xlim([0, 1])
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, results_df[metric])):
        ax.text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.suptitle('Performance Metrics Comparison Across All Methods', 
             y=1.02, fontsize=16, fontweight='bold')
plt.show()


### 6.3 Timing Comparison


In [ ]:
# Plot timing comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Extraction time
bars1 = ax1.barh(results_df['Method'], results_df['Extraction Time'], 
                 color='steelblue', alpha=0.8)
ax1.set_xlabel('Time (seconds)', fontsize=12)
ax1.set_title('Feature Extraction Time', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars1, results_df['Extraction Time'])):
    ax1.text(val + max(results_df['Extraction Time']) * 0.02, i, 
            f'{val:.2f}s', va='center', fontsize=10, fontweight='bold')

# Training time
bars2 = ax2.barh(results_df['Method'], results_df['Training Time'], 
                 color='coral', alpha=0.8)
ax2.set_xlabel('Time (seconds)', fontsize=12)
ax2.set_title('SVM Training Time', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars2, results_df['Training Time'])):
    ax2.text(val + max(results_df['Training Time']) * 0.02, i, 
            f'{val:.2f}s', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


### 6.4 Overall Summary


In [ ]:
# Find best method for each metric
print("="*80)
print("BEST METHODS BY METRIC")
print("="*80)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    best_idx = results_df[metric].idxmax()
    best_method = results_df.loc[best_idx, 'Method']
    best_value = results_df.loc[best_idx, metric]
    print(f"{metric:12s}: {best_method:30s} ({best_value:.4f})")
print("="*80)

# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Average Accuracy:  {results_df['Accuracy'].mean():.4f} ± {results_df['Accuracy'].std():.4f}")
print(f"Average Precision: {results_df['Precision'].mean():.4f} ± {results_df['Precision'].std():.4f}")
print(f"Average Recall:    {results_df['Recall'].mean():.4f} ± {results_df['Recall'].std():.4f}")
print(f"Average F1-Score:  {results_df['F1-Score'].mean():.4f} ± {results_df['F1-Score'].std():.4f}")
print("="*80)
